# <h1 align="center"> Filragem de SNPs utilizando GATK orientado por genoma de ref. em *Dipteryx alata* Vogel.

## <h2 align="justify"><span style="color: blue;">Esse script foi desenvolvido no contexto da dissertação de mestrado do aluno: Artur Gabriel Rodrigues Silva, no programa de pós graduação em Genética e Melhoramento de Plantas UFG (2025-2026), sob a orientação da professora Thannya Nascimento Soares, e coorientação de Marco Aurélio Caldas de Pinho Pessoa Filho da Embrapa Recursos Genéticos e Biotecnologia, responsável pelo desenvolvimento do script em questão, com algumas adequações do aluno, o script utiliza uma abordagem combinada do worflow RIG e FreeBayes.</span>

## Descrição do projeto

<h3 align="justify">A amostragem será composta por 24 populações de Dipteryx alata Vogel. representadas por um indivíduo cada, sequenciados em  sequenciados em lcWGS (9x~14x). As populações são pertencentes ao banco de germoplasma da UFG. Os dados de sequenciamento foram obtidos a partir de outros projetos realizados anteriormente, o sequenciamento foi realizado em Illumina NextSeq 1000, no Laboratório de Imunologia de Transplantes de Goiás (HLAGyn), o cartucho foi identificado como “NextSeq P2 600 ciclos”, a estimativa era de gerar 800 milhões de reads paired-end no total, que dividindo por 24 indivíduos esperava-se 33,33 milhões de reads por amostra, outra estimativa esperada era de gerar 240 Gb de dados, que dividindo pelo número de indivíduos esperava-se 10 Gb por amostra. O genoma de referência está montado em 8 cromossomos e 57 contigs.

## <h3 align="justify">Observação: O script foi gerado em um Jupyter Notebook, nos chunks de comando o identificadores %%bash são para comandos bash e %%R para comandos no R. Toda a análise será executada dentro de um Docker, dentro de um Tmux, dentro de um servidor, organize previamente o seu diretório contendo os arquivos brutos de sequenciamento, o seu genoma de referência, e um ou mais arquivos para *output* das análises para melhor organização.

### 0. Baixar a imagem oficial do GATK 4 (versão4.6.1.0 - Current) no Docker
O Image docker isola os programas e arquivos em um espaço dentro do servidor 

In [ ]:
%%bash

    docker pull broadinstitute/gatk

### 0.1 Inicie o docker com o volume montado para permitir o acesso e a gravação de dados (ajuste o caminho do diretório conforme a estrutura do seu servidor)
Você espelha temporariamente a sua pasta [/media/lgbio-nas1/artur.silva/data/projects/baru] dentro do conteiner criado [/gatk/my_data] com a imagem do GATK [-it broadinstitute/gatk]. Esse tipo de docker é temporário, é aconselhado a realizar dentro de um tmux para manter salvo.

In [ ]:
%%bash

	docker run -v /media/lgbio-nas1/artur.silva/data/projects/baru:/gatk/my_data -it broadinstitute/gatk
	apt-get update
	apt-get install bwa

### 1. Pré-processamento dos reads e mapeamento à referência
Acessa a pasta analysis dentro do conteiner. Cria um atalho chamado "genome.fasta" que referencia: [reference/Embrapa_UFG_Dala_nuclear_0.9.fa]. Em seguidaa cria índice a partir do arquivo genome.fasta para acelerar o processo de alinhamento com [BWA], -a bwtsw define o algorítmo de indexação. O comando faidx indexa um arquivo FASTA criando um arquivo .fai para ser utilizado pelo [SAMTOOLs]. O arquivo .dict será utilizado pelo [GATK]

In [ ]:
%%bash

	cd /gatk/my_data/analysis
	ln -s ../reference/Embrapa_UFG_Dala_nuclear_0.9.fa genome.fasta
	bwa index -a bwtsw genome.fasta
	samtools faidx genome.fasta
	gatk CreateSequenceDictionary -R genome.fasta -O genome.dict

O bioawk extrai os nomes de todos os contigs de um arquivo FASTA (genome.fasta) e salva esses nomes em um arquivo chamado contigs.list

In [ ]:
%%bash

	apt-get update && apt-get install -y git build-essential bison
	cd /tmp
	git clone https://github.com/lh3/bioawk.git
	cd bioawk
	make
	cp bioawk /usr/local/bin/
	bioawk -h
	cd /gatk/my_data/analysis
	bioawk -c fastx '{print $name}' genome.fasta > contigs.list

Gerando arquivo uBAM dos FASTQs (que saem do sequenciador) e adicionando as informações de cada grupo.
Atribuindo  infos: seq foward e reverse; nome do output; nome do grupo; id do individuo; nome da biblioteca a ser abastecida; dados de seq.; data da corrida e o log gerado. 

In [ ]:
%%bash

	gatk FastqToSam \
	-FASTQ ../sequencing/A01_L1_ds.c7dcdd780c7c45fab505cf673d5242f3/A01_S1_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/A01_L1_ds.c7dcdd780c7c45fab505cf673d5242f3/A01_S1_L001_R2_001.fastq.gz \
	-OUTPUT A01_fastqtosam.bam \
	-READ_GROUP_NAME A01 \
	-SAMPLE_NAME DalCGUFG_3.14 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> A01_fastqtosam.log 2>&1
	
	gatk FastqToSam \
	-FASTQ ../sequencing/A03_L1_ds.0e01508e1c4446e69e9a75474a373363/A03_S15_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/A03_L1_ds.0e01508e1c4446e69e9a75474a373363/A03_S15_L001_R2_001.fastq.gz \
	-OUTPUT A03_fastqtosam.bam \
	-READ_GROUP_NAME A03 \
	-SAMPLE_NAME DalCGUFG_7.38 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> A03_fastqtosam.log 2>&1 &

	gatk FastqToSam \
	-FASTQ ../sequencing/A04_L1_ds.86de693b63cf45c5adefcf894022c899/A04_S23_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/A04_L1_ds.86de693b63cf45c5adefcf894022c899/A04_S23_L001_R2_001.fastq.gz \
	-OUTPUT A04_fastqtosam.bam \
	-READ_GROUP_NAME A04 \
	-SAMPLE_NAME DalCGUFG_1.2Rep \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> A04_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/B02_L1_ds.3d32ff8d5914448bbdbd39a3460c14fd/B02_S8_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/B02_L1_ds.3d32ff8d5914448bbdbd39a3460c14fd/B02_S8_L001_R2_001.fastq.gz \
	-OUTPUT B02_fastqtosam.bam \
	-READ_GROUP_NAME B02 \
	-SAMPLE_NAME DalCGUFG_4.22 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> B02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/B03_L1_ds.817ca1e196e94faf8032b3e68ee928fc/B03_S16_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/B03_L1_ds.817ca1e196e94faf8032b3e68ee928fc/B03_S16_L001_R2_001.fastq.gz \
	-OUTPUT B03_fastqtosam.bam \
	-READ_GROUP_NAME B03 \
	-SAMPLE_NAME DalCGUFG_8.44 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> B03_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/B04_L1_ds.cf1a7ef74e5c4dee97de993015ab03eb/B04_S24_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/B04_L1_ds.cf1a7ef74e5c4dee97de993015ab03eb/B04_S24_L001_R2_001.fastq.gz \
	-OUTPUT B04_fastqtosam.bam \
	-READ_GROUP_NAME B04 \
	-SAMPLE_NAME DalCGUFG_10.55Rep \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> B04_fastqtosam.log 2>&1 &
	
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) C01_L1_ds.6957fdf4e9e340e6add927afa31690be'/C01_S2_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) C01_L1_ds.6957fdf4e9e340e6add927afa31690be'/C01_S2_L001_R2_001.fastq.gz \
	-OUTPUT C01_fastqtosam.bam \
	-READ_GROUP_NAME C01 \
	-SAMPLE_NAME DalCGUFG_16.94 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> C01_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) C02_L1_ds.163fbf3f92a24fcfb6c2789646d1e345'/C02_S9_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) C02_L1_ds.163fbf3f92a24fcfb6c2789646d1e345'/C02_S9_L001_R2_001.fastq.gz \
	-OUTPUT C02_fastqtosam.bam \
	-READ_GROUP_NAME C02 \
	-SAMPLE_NAME DalCGUFG_2.7 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> C02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) C03_L1_ds.a0f6bebc713e407ca9c311188919d151'/C03_S17_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) C03_L1_ds.a0f6bebc713e407ca9c311188919d151'/C03_S17_L001_R2_001.fastq.gz \
	-OUTPUT C03_fastqtosam.bam \
	-READ_GROUP_NAME C03 \
	-SAMPLE_NAME DalCGUFG_20.117 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> C03_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) D01_L1_ds.287a062b91ce48adb531b6d66aff00d8'/D01_S3_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) D01_L1_ds.287a062b91ce48adb531b6d66aff00d8'/D01_S3_L001_R2_001.fastq.gz \
	-OUTPUT D01_fastqtosam.bam \
	-READ_GROUP_NAME D01 \
	-SAMPLE_NAME DalCGUFG_23.136 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> D01_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) D02_L1_ds.21bce54efdc946698e799b16cbf8504e'/D02_S10_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) D02_L1_ds.21bce54efdc946698e799b16cbf8504e'/D02_S10_L001_R2_001.fastq.gz \
	-OUTPUT D02_fastqtosam.bam \
	-READ_GROUP_NAME D02 \
	-SAMPLE_NAME DalCGUFG_14.80 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> D02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) D03_L1_ds.7b6960bcb2bc4cff9faa336369286904'/D03_S18_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) D03_L1_ds.7b6960bcb2bc4cff9faa336369286904'/D03_S18_L001_R2_001.fastq.gz \
	-OUTPUT D03_fastqtosam.bam \
	-READ_GROUP_NAME D03 \
	-SAMPLE_NAME DalCGUFG_19.133 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> D03_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) E01_L1_ds.59e9aa4f01ee47b793d2c58775aebee8'/E01_S4_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) E01_L1_ds.59e9aa4f01ee47b793d2c58775aebee8'/E01_S4_L001_R2_001.fastq.gz \
	-OUTPUT E01_fastqtosam.bam \
	-READ_GROUP_NAME E01 \
	-SAMPLE_NAME DalCGUFG_22.127 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> E01_fastqtosam.log 2>&1 &
	 
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) E02_L1_ds.536706eeceaa4507b4fb275ed1878662'/E02_S11_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) E02_L1_ds.536706eeceaa4507b4fb275ed1878662'/E02_S11_L001_R2_001.fastq.gz \
	-OUTPUT E02_fastqtosam.bam \
	-READ_GROUP_NAME E02 \
	-SAMPLE_NAME DalCGUFG_17.98 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> E02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) E03_L1_ds.3360eeafe052475196eb6bcc32ad8908'/E03_S19_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) E03_L1_ds.3360eeafe052475196eb6bcc32ad8908'/E03_S19_L001_R2_001.fastq.gz \
	-OUTPUT E03_fastqtosam.bam \
	-READ_GROUP_NAME E03 \
	-SAMPLE_NAME DalCGUFG_18.105 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> E03_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) F01_L1_ds.cae7f69d0fac459383850fff40547848'/F01_S5_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) F01_L1_ds.cae7f69d0fac459383850fff40547848'/F01_S5_L001_R2_001.fastq.gz \
	-OUTPUT F01_fastqtosam.bam \
	-READ_GROUP_NAME F01 \
	-SAMPLE_NAME DalCGUFG_15.88 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> F01_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) F02_L1_ds.4f90c612fb9641d793c1e939f7a075e9'/F02_S12_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) F02_L1_ds.4f90c612fb9641d793c1e939f7a075e9'/F02_S12_L001_R2_001.fastq.gz \
	-OUTPUT F02_fastqtosam.bam \
	-READ_GROUP_NAME F02 \
	-SAMPLE_NAME DalCGUFG_13.73 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> F02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) F03_L1_ds.54376380824a43368235ae9190fbb83a'/F03_S20_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) F03_L1_ds.54376380824a43368235ae9190fbb83a'/F03_S20_L001_R2_001.fastq.gz \
	-OUTPUT F03_fastqtosam.bam \
	-READ_GROUP_NAME F03 \
	-SAMPLE_NAME DalCGUFG_5.26 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> F03_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) G01_L1_ds.ff0cf85f24ab4b6db7348af5e709f086'/G01_S6_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) G01_L1_ds.ff0cf85f24ab4b6db7348af5e709f086'/G01_S6_L001_R2_001.fastq.gz \
	-OUTPUT G01_fastqtosam.bam \
	-READ_GROUP_NAME G01 \
	-SAMPLE_NAME DalCGUFG_6.34 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> G01_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) G02_L1_ds.0792a546e29b4095aa865f3a93f74f8e'/G02_S13_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) G02_L1_ds.0792a546e29b4095aa865f3a93f74f8e'/G02_S13_L001_R2_001.fastq.gz \
	-OUTPUT G02_fastqtosam.bam \
	-READ_GROUP_NAME G02 \
	-SAMPLE_NAME DalCGUFG_11.66 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> G02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) G03_L1_ds.a5e032c313734eb4a79c8e5684591ade'/G03_S21_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) G03_L1_ds.a5e032c313734eb4a79c8e5684591ade'/G03_S21_L001_R2_001.fastq.gz \
	-OUTPUT G03_fastqtosam.bam \
	-READ_GROUP_NAME G03 \
	-SAMPLE_NAME DalCGUFG_25.147 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> G03_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) H01_L1_ds.8ff55fa714944b6e990d9641c6e97f26'/H01_S7_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) H01_L1_ds.8ff55fa714944b6e990d9641c6e97f26'/H01_S7_L001_R2_001.fastq.gz \
	-OUTPUT H01_fastqtosam.bam \
	-READ_GROUP_NAME H01 \
	-SAMPLE_NAME DalCGUFG_21.121 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> H01_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) H02_L1_ds.162c4b74853f4bec860f3e34ebe404fd'/H02_S14_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) H02_L1_ds.162c4b74853f4bec860f3e34ebe404fd'/H02_S14_L001_R2_001.fastq.gz \
	-OUTPUT H02_fastqtosam.bam \
	-READ_GROUP_NAME H02 \
	-SAMPLE_NAME DalCGUFG_12.67 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> H02_fastqtosam.log 2>&1 &
	
	gatk FastqToSam \
	-FASTQ ../sequencing/'(copy) H03_L1_ds.bd31e203bcf24d91b8addcb492d0e9e6'/H03_S22_L001_R1_001.fastq.gz \
	-FASTQ2 ../sequencing/'(copy) H03_L1_ds.bd31e203bcf24d91b8addcb492d0e9e6'/H03_S22_L001_R2_001.fastq.gz \
	-OUTPUT H03_fastqtosam.bam \
	-READ_GROUP_NAME H03 \
	-SAMPLE_NAME DalCGUFG_9.50 \
	-LIBRARY_NAME Genomas_Baru_UFGO \
	-PLATFORM_UNIT AACWW3FM5 \
	-PLATFORM illumina \
	-SEQUENCING_CENTER HLAGyn \
	-RUN_DATE 2023-01-25 >> H03_fastqtosam.log 2>&1 &

Marcando os adaptadores Illumina (*processo em loop*)


In [ ]:
%%bash

	for file in *.bam
	do
		gatk MarkIlluminaAdapters \
		-I $file \
		-O ${file/%bam/markilluminaadapters.bam} \
		-M ${file/%bam/markilluminaadapters_metrics.txt}  &
	done

### 2. Pipeline para alinhar as reads ao genoma de referência usando BWA-MEM e mesclar com uBAM usando MergeBamAlignment
Fornece o uBAM original com metadados (não alinhado),fornece o uBAM com adaptadores marcados (mesmo conteúdo, mas com informações de adaptadores), e o pipeline extrai as leituras do uBAM marcado, alinha ao genoma de referência com BWA (só aceita input em FASTQ), depois faz a fusão entre o alinhamento e o uBAM original para preservar os metadados. Resultado: um BAM final alinhado, com os metadados originais, adaptadores marcados e alinhados a referência identificado por: [piped.bam]

Como você está em um 'docker', não há problemas em só copiar e colar o código, mas se quiser deixar um script salvo e só executar, basta criar o script com [nano (nomedoscript.sh)]. Uma vez criado não esqueça do cabeçalho antes do comando a seguir: #!/bin/bash

In [ ]:
%%bash

	set -euo pipefail
	
	file1=(*.markilluminaadapters.bam) 
	file2=(*_fastqtosam.bam) 
	
	# add a check to ensure lenghts of file1 and file2 are equal, to avoid mismatches
	
	if [ "${#file1[@]}" -ne "${#file2[@]}" ]; then
    echo "Error: file1 and file2 arrays have different lengths" >&2
    exit 1
	fi

	for ((i=0;i<${#file1[@]};i++)); do     
    	gatk SamToFastq \
    	-I "${file1[i]}" \
    	-F /dev/stdout \
    	--CLIPPING_ATTRIBUTE XT \
    	--CLIPPING_ACTION 2 \
    	--INTERLEAVE true \
    	-NON_PF true \
    	--TMP_DIR ./db | \
    	
    	bwa mem -M -t 16 -p genome.fasta /dev/stdin | \
    	
    	gatk MergeBamAlignment \
    	--ALIGNED_BAM /dev/stdin \
    	--UNMAPPED_BAM "${file2[i]}" \
    	--OUTPUT "${file1[i]/%bam/piped.bam}" \
    	-R genome.fasta \
    	--CREATE_INDEX true \
    	--ADD_MATE_CIGAR true \
    	--CLIP_ADAPTERS false \
    	--CLIP_OVERLAPPING_READS true \
    	--INCLUDE_SECONDARY_ALIGNMENTS true \
    	--MAX_INSERTIONS_OR_DELETIONS -1 \
    	--PRIMARY_ALIGNMENT_STRATEGY MostDistant \
    	--ATTRIBUTES_TO_RETAIN XS \
    	--TMP_DIR ./db 
	done

Libere espaço de armazenamento em nosso sistema de arquivos excluindo os arquivos originais com os quais começamos, o uBAM e o uBAMTXT

(Opcional) Anotando a cobertura de cada indivíduo

In [ ]:
%%bash

	for bam in *.piped.bam
	do
		if [[ -f "$bam" ]]; then
			cobertura=$(samtools depth "$bam" | awk '{sum+=$3} END {if (NR>0) print sum/NR; else print 0}')
			nome=$(basename "$bam" .piped.bam)
			echo -e "${nome}\t${cobertura}" >> cobertura_media.tsv
		fi
	done

### 3. Marcar as duplicatas
Marca duplicatas para evitar viés de cobertura causado por cópias artificiais (duplicatas PCR), melhorar a precisão na detecção de variantes, reduz falsos positivos e preserva apenas uma cópia representativa por fragmento original, mantendo a integridade dos dados

In [ ]:
%%bash

	for file in *.piped.bam
	do
	gatk MarkDuplicates \
		-I $file \
		-O ${file/%fastqtosam.markilluminaadapters.piped.bam/marked_duplicates.bam} \
		-M ${file/%fastqtosam.markilluminaadapters.piped.bam/dup_metrics.txt} \
		--OPTICAL_DUPLICATE_PIXEL_DISTANCE 2500 
	done

Agora necessitamos dos [*.bai] para etapa 4.

In [ ]:
%%bash

	for file in *_marked_duplicates.bam
	do
		samtools index $file
	done

### 4. Naive pipeline - Step 1 - Chamada de variantes para construir um banco de dados de verossimilhança
Step 1: HaplotypeCaller ERC GVCF [g.vcf.gz] (O HaplotypeCaller reconstrói haplótipos localmente em vez de chamar variantes base por base, o que melhora a precisão, especialmente em regiões complexas. O parâmetro -ERC GVCF ativa o modo de emissão de variantes e regiões não variantes, gerando um gVCF individual que pode ser posteriormente combinado com outros para genotipagem conjunta)

Atenção ! Limitação de núcleos por usuário do seu servidor, utilizou-se 1 núcleos para cada nesse caso 

In [ ]:
%%bash

	for file in *_marked_duplicates.bam
	do
		nohup gatk HaplotypeCaller \
		-I $file \
		-O ${file/%marked_duplicates.bam/step1.g.vcf.gz} \
		-R genome.fasta \
		-ploidy 2 \
		--native-pair-hmm-threads 1 \
		-ERC GVCF > ${file/%marked_duplicates.bam/_hc.step1.log} &
	done

Criando o banco de verossimilhanças para a genotipagem conjunta

Atenção ! É necessário um diretório temporário para essa etapa e as seguintes, para isso crie um diretório temporário na pasta que está realizando suas análises [mkdir ./db], e confira a esse diretório todas as permissões necessárias [chmod 777 ./db]

In [ ]:
%%bash

	gatk GenomicsDBImport \
	-R genome.fasta \
	-V A01_step1.g.vcf.gz \
	-V A03_step1.g.vcf.gz \
	-V A04_step1.g.vcf.gz \
	-V B02_step1.g.vcf.gz \
	-V B03_step1.g.vcf.gz \
	-V B04_step1.g.vcf.gz \
	-V C01_step1.g.vcf.gz \
	-V C02_step1.g.vcf.gz \
	-V C03_step1.g.vcf.gz \
	-V D01_step1.g.vcf.gz \
	-V D02_step1.g.vcf.gz \
	-V D03_step1.g.vcf.gz \
	-V E01_step1.g.vcf.gz \
	-V E02_step1.g.vcf.gz \
	-V E03_step1.g.vcf.gz \
	-V F01_step1.g.vcf.gz \
	-V F02_step1.g.vcf.gz \
	-V F03_step1.g.vcf.gz \
	-V G01_step1.g.vcf.gz \
	-V G02_step1.g.vcf.gz \
	-V G03_step1.g.vcf.gz \
	-V H01_step1.g.vcf.gz \
	-V H02_step1.g.vcf.gz \
	-V H03_step1.g.vcf.gz \
	--genomicsdb-workspace-path my_database_baru \
	--tmp-dir ./db \
	-L contigs.list \
	--max-num-intervals-to-import-in-parallel 24

Dividindo genoma em 32 pedaços [interval_list] para processamento paralelo na etapa seguinte

In [ ]:
%%bash

	gatk SplitIntervals -R genome.fasta -L contigs.list -O interval-files --scatter-count 32 --subdivision-mode INTERVAL_COUNT

Confira se foram gerados 32 intervalos

In [ ]:
%%bash

	ls interval-files/*.interval_list | wc -l

Realizando a genotipagem conjunta, usando os [*.g.vcf.gz] já importados no banco de verossimilhanças

In [ ]:
%%bash

	for file in interval-files/*.interval_list
	do
		nohup gatk GenotypeGVCFs \
		-R genome.fasta \
		-V gendb://my_database_baru \
		-O "$(basename "$file" -scattered.interval_list).step1.raw.snps.vcf.gz" \
		--tmp-dir ./db \
		-L $file > "$(basename "$file" -scattered.interval_list).genotypeGVCFS.step1.log" &  
	done

Crie um arquivo chamado [inputs.raw.vcf.list] para juntar os VCF

In [ ]:
%%bash

	ls *.step1.raw.snps.vcf.gz | sort > inputs.raw.vcf.list

Junta os VCF

In [ ]:
%%bash

	gatk GatherVcfs \
	-R genome.fasta \
	-I inputs.raw.vcf.list \
	-O allsamples.step1.raw.snps.vcf.gz

### 5. Naive pipeline - Hard-filter para BQSR
Essa etapa realiza hard-filtering, um processo determinístico que aplica critérios fixos e objetivos de qualidade em SNPs e INDELs (como QD - qualidade por profundidade de leitura, MQ - qualidade de mapeamento das leituras, FS - Teste Fisher para viés de distribuição das leituras), removendo variantes de baixa confiança com base em métricas estatísticas e gerando um VCF limpo e confiável para ser usado na recalibração da qualidade das bases (BQSR)

Permitindo acesso rápido por posição genômica: necessário para muitos comandos do GATK. O índice permite que ferramentas como o GATK, IGV ou bcftools acessam rapidamente regiões específicas do arquivo, sem precisar ler tudo.

In [ ]:
%%bash

	gatk IndexFeatureFile \
     -I allsamples.step1.raw.snps.vcf.gz

Separa SNPs e INDELs e indexa os arquivos

In [ ]:
%%bash

	gatk SelectVariants \
    -V allsamples.step1.raw.snps.vcf.gz \
    -select-type SNP \
    -O allsamples.step1.raw.snps4bqsr.vcf.gz

In [ ]:
%%bash

	gatk SelectVariants \
    -V allsamples.step1.raw.snps.vcf.gz \
    -select-type INDEL \
    -O allsamples.step1.raw.indels4bqsr.vcf.gz

Gerarando tabelas para plotar as métricas no R. Isso ajuda a definir valores de corte apropriados (*thresholds*) para filtrar variantes.

In [ ]:
%%bash

	gatk VariantsToTable \
	-V allsamples.step1.raw.snps4bqsr.vcf.gz \
	-F CHROM -F POS \
	-F DP -F QD -F QUAL -F SOR -F FS -F MQ \
	-F BaseQRankSum -F MQRankSum -F ReadPosRankSum \
	-O allsamples.step1.raw.snps4bqsr.table
	
	gatk VariantsToTable \
	-V allsamples.step1.raw.indels4bqsr.vcf.gz \
	-F CHROM -F POS -F QD -F QUAL -F SOR -F FS -F MQ -F MQRankSum -F ReadPosRankSum \
	-O allsamples.step1.raw.indels4bqsr.table

Guia para estudo dos filtros: https://github.com/broadinstitute/gatk-docs/blob/master/gatk3-methods-and-algorithms/Understanding_and_adapting_the_generic_hard-filtering_recommendations.md 

Filtrando SNPs

In [ ]:
%%bash

	gatk VariantFiltration     \
	-V allsamples.step1.raw.snps4bqsr.vcf.gz     \
	--filter-name "QD2" -filter "QD < 2.0" \
	--filter-name "SOR2" -filter "SOR > 2.0"      \
	--filter-name "FS50" -filter "FS > 50.0"      \
	--filter-name "MQ50" -filter "MQ < 50.0"      \
	--filter-name "MQRankSum-5" -filter "MQRankSum < -5.0"      \
	--filter-name "ReadPosRankSum-5" -filter "ReadPosRankSum < -5.0"      \
	-O allsamples.step1.filtered.snps4bqsr.vcf.gz

In [ ]:
Filtrando INDELs

In [ ]:
%%bash

	gatk VariantFiltration \
	-V allsamples.step1.raw.indels4bqsr.vcf.gz \
	-filter "QD < 2.0" --filter-name "QD2" \
	-filter "FS > 200.0" --filter-name "FS200" \
	-filter "SOR > 3.0" --filter-name "SOR3" \
	-filter "MQ < 50.0" --filter-name "MQ50" \
	-filter "MQRankSum < -5.0" --filter-name "MQRankSum-50" \
	-filter "ReadPosRankSum < -5.0" --filter-name "ReadPosRankSum-5" \
	-O allsamples.step1.filtered.indels4bqsr.vcf.gz

Junta os SNPs e INDELs bons de volta em um único VCF

In [ ]:
%%bash

	gatk MergeVcfs\
	-I allsamples.step1.filtered.snps4bqsr.vcf.gz \
	-I allsamples.step1.filtered.indels4bqsr.vcf.gz \
	-O allsamples.step1.filtered.allvariants4bqsr.vcf.gz

Remove as variantes que falharam nos filtros anteriores, gerando um VCF limpo de variantes sensíveis, para entrada para o BQSR.

In [ ]:
%%bash

	gatk SelectVariants --exclude-filtered \
	-R genome.fasta \
	-V allsamples.step1.filtered.allvariants4bqsr.vcf.gz \
	-O allsamples.step1.filtered.clean.allvariants4bqsr.vcf.gz

### 6. Naive pipeline - BQSR1
Gerando uma tabela de recalibração para o Base Quality Score Recalibration (BQSR) que descreve padrões de erro nas qualidades de base (ex: erros por ciclo, por contexto de base etc.)

In [ ]:
%%bash

	for file in *_marked_duplicates.bam
	do
		nohup gatk BaseRecalibrator \
		-R genome.fasta \
		-I $file \
		--known-sites allsamples.step1.filtered.clean.allvariants4bqsr.vcf.gz \
		-O ${file/%marked_duplicates.bam/bqsr_recal.table} > ${file/%marked_duplicates.bam/baserecal.log} & 
	done

Aplicando BQSR

In [ ]:
%%bash

	set -o pipefail
	file1=(*_marked_duplicates.bam)
	file2=(*_bqsr_recal.table)
	
	for ((i=0;i<=${#file1[@]};i++)); do 
		nohup gatk ApplyBQSR \
		-R genome.fasta \
		-I "${file1[i]}" \
		--static-quantized-quals 10 --static-quantized-quals 20 --static-quantized-quals 30 \
		-bqsr "${file2[i]}" \
		-O "${file1[i]/%marked_duplicates.bam/recalibrated.bam}" > "${file1[i]/%marked_duplicates.bam/applybqsr.log}" &
	done

As qualidades de base são ajustadas com base na tabela gerada, removendo viés sistemático, --static-quantized-quals reduz o número de níveis de qualidade possíveis (aumenta a compressão e performance sem perder precisão)

### 7. Naive pipeline - Rodar FreeBayes com os BAMs recalibrados
Lista todos os BAMs recalibrados (da etapa de BQSR) e salva no formato .fofn (file of file names), necessário para o FreeBayes. Será criado e ativado um ambiente Conda limpo para rodar o FreeBayes, garantindo controle sobre versões. 

In [ ]:
%%bash

	ls *_recalibrated.bam > recalibrated.bams.fofn
	mkdir freebayes && cd freebayes
	conda create -n freebayes -c conda-forge -c bioconda freebayes=1.3.6
	conda activate freebayes

Executa o FreeBayes em paralelo, quebrando o genoma em 30 regiões e usando 30 threads para acelerar a análise em todos os BAMs.

In [ ]:
%%bash

	freebayes-parallel <(fasta_generate_regions.py genome.fasta.fai --chunks 50) 16    \
	-f genome.fasta \
	-g 200 \
	--bam-list recalibrated.bams.fofn  > freebayes/freebayes.raw.variants.vcf

Instalando rtg tools no Docker

In [ ]:
%%bash

	cd ~
	wget https://github.com/RealTimeGenomics/rtg-tools/releases/download/3.12.1/rtg-tools-3.12.1-linux-x64.zip
	unzip rtg-tools-3.12.1-linux-x64.zip

Gera estatísticas básicas das variantes com rtg-tools, que dá uma visão geral de qualidade

In [ ]:
%%bash

    ~/rtg-tools-3.12.1/rtg vcfstats freebayes.raw.variants.vcf > freebayes.raw.variants.rtg.stats
    bgzip freebayes.raw.variants.vcf
    tabix freebayes.raw.variants.vcf.gz

Extrai valores de QUAL e DP das variantes (https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9163752/)

In [ ]:
%%bash

	bcftools query --include 'QUAL>20' --format '%QUAL\n' freebayes.raw.variants.vcf.gz > quals.tmp
	bcftools query --include 'QUAL>20' --format '%INFO/DP\n' freebayes.raw.variants.vcf.gz | awk -F "," '{print $1}' > dps.tmp

Depois, você roda R para calcular os quantis (percentis) e definir limites máximos adequados para filtrar outliers:

In [ ]:
%%R
    R
	 dps <- as.numeric(readLines("dps.tmp")) 
	 quals <- as.numeric(readLines("quals.tmp"))
	 qudp <- unname(round(quantile(dps,0.99)))
	 ququ <- unname(round(quantile(quals,0.99)))
	 write(qudp,file("dpt.tmp"))
	 write(ququ,file("qut.tmp"))

Aplica os filtros, decompõe e normaliza variantes complexas. Foi necessário fazer downgrade do vcflib para a versão 1.0.1 no ambiente Conda do FreeBayes para que isso funcionasse.

In [ ]:
%%bash

	conda create -n vcflib-env python=3.10 vcflib=1.0.1 -c bioconda -c conda-forge 
	conda activate vcflib-env
	less qut.tmp
	less dpt.tmp
	bcftools view --include 'QUAL>20 & INFO/DP>10 & QUAL<3371 & INFO/DP<193 & (QUAL/(INFO/DP))>2' freebayes.raw.variants.vcf.gz | vcfallelicprimitives -kg | bcftools norm --fasta-ref ../genome.fasta --output-type z --output freebayes.filtered.norm.vcf.gz

	zgrep -v "^#" freebayes.filtered.norm.vcf.gz | wc -l

Encontrando SNPs em comúm com GATK-step1 e Freebayes-1 datasets, utilizando `bedtools`, obtendo nos variantes específicas para VQSR

In [ ]:
%%bash

	conda create -n bedtools-env bedtools -c bioconda -c conda-forge
	conda activate bedtools-env
	bedtools intersect -sorted -g genome.fasta.fai -header -u -a allsamples.step1.filtered.clean.allvariants4bqsr.vcf.gz -b freebayes/freebayes.filtered.norm.vcf.gz -wa > gatk.step1.freebayes.intersect.vcf

	zgrep -v "^#" gatk.step1.freebayes.intersect.vcf | wc -l

Seleciona variantes bialélicas (para análise de populações, GWAS, etc.) com MAF > 0.05

In [ ]:
%%bash

	gatk IndexFeatureFile -I gatk.step1.freebayes.intersect.vcf
	gatk SelectVariants -R genome.fasta -V gatk.step1.freebayes.intersect.vcf --restrict-alleles-to BIALLELIC --exclude-non-variants -select "AF > 0.05" -O gatk.step1.freebayes.intersect.biallelic.maf0.05.vcf.gz

	zgrep -v "^#" gatk.step1.freebayes.intersect.biallelic.maf0.05.vcf.gz | wc -l

Reduz o número de variantes, mantendo apenas 1 variante a cada 150 pares de bases (pruning), para evitar viés por densidade e linkage.

In [ ]:
%%bash

	bcftools +prune -n 1 -w 150bp --AF-tag "0.05" gatk.step1.freebayes.intersect.biallelic.maf0.05.vcf.gz -Oz -o gatk.step1.freebayes.biallelic.maf0.05.prune150.vcf.gz
	gatk IndexFeatureFile -I gatk.step1.freebayes.biallelic.maf0.05.prune150.vcf.gz

	zgrep -v "^#" gatk.step1.freebayes.biallelic.maf0.05.prune150.vcf.gz | wc -l

### 8. Initial Informed Pipeline - Variant Quality Score Recalibration (VQSR1)
Converte o VCF para conter apenas a posição das variantes (SNPs), sem os dados de genótipo dos indivíduos.

In [ ]:
%%bash

	gatk  MakeSitesOnlyVcf \
	-I allsamples.step1.raw.snps4bqsr.vcf.gz \
	-O allsamples.step1.sitesonly.raw.snps4vqsr.vcf.gz

Utilizaremos como recurso o arquivo `gatk.step1.freebayes.biallelic.maf0.05.prune150.vcf.gz`. Esse conjunto tem variantes de alta qualidade (por estarem em comum no GATK e FreeBayes, bialélicas, com MAF > 5%, espaçadas a cada 150bp). O GATK entende que essas variantes são confiáveis (truth e training = true) e com um peso de confiança (prior = 7.0). Criando um modelo estatístico (mixture of Gaussians) com 6 componentes para classificar variantes com base em anotações como: MQ, QD, FS, SOR, DP, ReadPosRankSum, etc. Define os `tranches` de confiança a partir dos VQSLOD.

In [ ]:
%%bash

	gatk --java-options "-Xmx64g -Xms64g" VariantRecalibrator \
	-V allsamples.step1.sitesonly.raw.snps4vqsr.vcf.gz \
	--trust-all-polymorphic \
	-tranche 100.0 -tranche 99.9 -tranche 99.0 -tranche 97.5 -tranche 95.0 -tranche 90.0 \
	-an MQ -an MQRankSum -an QD -an ReadPosRankSum -an FS -an SOR -an DP \
	-mode SNP \
	--max-gaussians 6 \
	-resource:step1intersect,known=false,training=true,truth=true,prior=7.0 gatk.step1.freebayes.biallelic.maf0.05.prune150.vcf.gz \
	-O allsamples.step1.snps.recal.vcf.gz \
	--tranches-file aallsamples.step1.snps.tranches \
	--rscript-file allsamples.step1.vqsr.plots.R

Aplicando VQSR

In [ ]:
%%bash

	gatk --java-options "-Xmx64g -Xms64g" ApplyVQSR \
	-V allsamples.step1.raw.snps.vcf.gz \
	--recal-file allsamples.step1.snps.recal.vcf.gz \
	--tranches-file aallsamples.step1.snps.tranches \
	--truth-sensitivity-filter-level 90.0 \
	--create-output-variant-index true \
	-mode SNP \
	-O allsamples.step1.recalibrated.snps.vcf.gz

Selecionando variantes de tranche 90

In [ ]:
%%bash

	gatk SelectVariants \
	--exclude-filtered \
	-R genome.fasta \
	-V allsamples.step1.recalibrated.snps.vcf.gz \
	-O allsamples.step1.tranche90.PASS.snps.vcf.gz 
	zgrep -v "^#" allsamples.step1.tranche90.PASS.snps.vcf.gz | wc -l

### 9. Informed pipeline, Base Quality Score Recalibration 2 (BQSR2) de reBAMs utilizandoa as variantes recalibradas da etapa anterior como "sítios conhecidos" 
Gera uma tabela de recalibração para o Base Quality Score Recalibration (BQSR2), a saída é um arquivo `.table` com informações sobre erros sistemáticos nas qualidades das bases, que será usada para ajustar os scores de base.

In [ ]:
%%bash

	for file in *_recalibrated.bam
	do
		nohup gatk BaseRecalibrator \
		-R genome.fasta \
		-I $file \
		--known-sites allsamples.step1.tranche90.PASS.snps.vcf.gz \
		-O ${file/%recalibrated.bam/bqsr_step2_recal.table} > ${file/%recalibrated.bam/baserecal_step2.log} & 
	done

Aplicando a tabela

In [ ]:
%%bash

        set -o pipefail
        file1=(*_marked_duplicates.bam)
        file2=(*_bqsr_step2_recal.table)

        for ((i=0;i<${#file1[@]};i++)); do
                nohup gatk ApplyBQSR \
                -R genome.fasta \
                -I "${file1[i]}" \
                --static-quantized-quals 10 --static-quantized-quals 20 --static-quantized-quals 30 \
                -bqsr "${file2[i]}" \
                -O "${file1[i]/%marked_duplicates.bam/recalibrated.step2.bam}" > "${file1[i]/%marked_duplicates.bam/applybqsr.step2.log}" &
        done

### 10. Informed pipeline - HaplotypeCaller e GenotypeGVCF nos BAMs re-calibrados
Segue a mesma lógica do item `###4`, agora com os BAMs recalibrados. Os BAMS foram primeiramente recalibrados por qualidade de base utilizando SNPs `hard filtered`, e depois utilizando VQSR SNPs. 

Step 2 HaplotypeCaller ERC GVCF 
Para 24 amostras, foi utilizado 1 núcleo para cada	

In [ ]:
%%bash

	for file in *_recalibrated.step2.bam
	do
		nohup gatk HaplotypeCaller \
		-I $file \
		-O ${file/%recalibrated.step2.bam/step2.g.vcf.gz} \
		-R genome.fasta \
		-ploidy 2 \
		--native-pair-hmm-threads 1 \
		-ERC GVCF > ${file/%recalibrated.step2.bam/hc.step2.log} &
	done

Combinando GVCFs 

In [ ]:
%%bash

	gatk GenomicsDBImport \
	-R genome.fasta \
	-V A01_step2.g.vcf.gz \
	-V A03_step2.g.vcf.gz \
	-V A04_step2.g.vcf.gz \
	-V B02_step2.g.vcf.gz \
	-V B03_step2.g.vcf.gz \
	-V B04_step2.g.vcf.gz \
	-V C01_step2.g.vcf.gz \
	-V C02_step2.g.vcf.gz \
	-V C03_step2.g.vcf.gz \
	-V D01_step2.g.vcf.gz \
	-V D02_step2.g.vcf.gz \
	-V D03_step2.g.vcf.gz \
	-V E01_step2.g.vcf.gz \
	-V E02_step2.g.vcf.gz \
	-V E03_step2.g.vcf.gz \
	-V F01_step2.g.vcf.gz \
	-V F02_step2.g.vcf.gz \
	-V F03_step2.g.vcf.gz \
	-V G01_step2.g.vcf.gz \
	-V G02_step2.g.vcf.gz \
	-V G03_step2.g.vcf.gz \
	-V H01_step2.g.vcf.gz \
	-V H02_step2.g.vcf.gz \
	-V H03_step2.g.vcf.gz \
	--genomicsdb-workspace-path my_database_baru_step2_new \
	--tmp-dir ./db \
	-L contigs.list \
	--max-num-intervals-to-import-in-parallel 24 

Divide o genoma em 32 pedaços (interval_list) para processamento paralelo na etapa seguinte

In [ ]:
%%bash

	gatk SplitIntervals -R genome.fasta -L contigs.list -O interval-files --scatter-count 32 --subdivision-mode INTERVAL_COUNT

Genotipagem conjunta

In [ ]:
%%bash
	
	for file in interval-files/*.interval_list; do      
		nohup gatk GenotypeGVCFs \
		-R genome.fasta \
		-V gendb://my_database_baru_step2_new \
		-O "$(basename "$file" -scattered.interval_list).step2.raw.variants.vcf.gz" \
		--tmp-dir ./db \
		-L $file > "$(basename "$file" -scattered.interval_list).genotypeGVCFS.step2.log" &  
		done

Crie um arquivo chamado: [inputs.raw.vcf.list]

In [ ]:
%%bash

	ls *.step2.raw.variants.vcf.gz | sort > inputs.step2.raw.vcf.list

Juntando VCFs

In [ ]:
%%bash

	gatk GatherVcfs \
	-R genome.fasta \
	-I inputs.step2.raw.vcf.list \
	-O allsamples.step2.raw.variants.vcf.gz

Indexando

In [ ]:
%%bash

	gatk IndexFeatureFile \
    -I allsamples.step2.raw.variants.vcf.gz

Separando SNPs 

In [ ]:
%%bash

	gatk SelectVariants \
    -V allsamples.step2.raw.variants.vcf.gz \
    -select-type SNP \
    -O allsamples.step2.raw.snps4bqsr.vcf.gz

Separando INDELs

In [ ]:
%%bash

	gatk SelectVariants \
    -V allsamples.step2.raw.variants.vcf.gz \
    -select-type INDEL \
    -O allsamples.step2.raw.indels4bqsr.vcf.gz

Gerando tabelas com as métricas para visualização no R

In [ ]:
%%bash

	gatk VariantsToTable \
	-V allsamples.step2.raw.snps4bqsr.vcf.gz \
	-F CHROM -F POS \
	-F DP -F QD -F QUAL -F SOR -F FS -F MQ \
	-F BaseQRankSum -F MQRankSum -F ReadPosRankSum \
	-O allsamples.step2.raw.snps4bqsr.table
	
	gatk VariantsToTable \
	-V allsamples.step2.raw.indels4bqsr.vcf.gz \
	-F CHROM -F POS -F QD -F QUAL -F SOR -F FS -F MQ -F MQRankSum -F ReadPosRankSum \
	-O allsamples.step2.raw.indels4bqsr.table

O script R `snp-quality-filters.R` contém estatísticas por variante para análise visual em R. Isso ajuda a definir valores de corte apropriados (thresholds) para filtrar variantes.

Filtrando SNPs baseado nos gráficos de densidade do R

In [ ]:
%%bash

	gatk VariantFiltration     \
	-V allsamples.step2.raw.snps4bqsr.vcf.gz     \
	--filter-name "QD2" -filter "QD < 2.0" \
	--filter-name "SOR2" -filter "SOR > 2.0"      \
	--filter-name "FS50" -filter "FS > 50.0"      \
	--filter-name "MQ50" -filter "MQ < 50.0"      \
	--filter-name "MQRankSum-5" -filter "MQRankSum < -5.0"      \
	--filter-name "ReadPosRankSum-5" -filter "ReadPosRankSum < -5.0"      \
	-O allsamples.step2.filtered.snps.vcf.gz

Filtrando INDELS

In [ ]:
%%bash

	gatk VariantFiltration \
	-V allsamples.step2.raw.indels4bqsr.vcf.gz \
	-filter "QD < 2.0" --filter-name "QD2" \
	-filter "FS > 200.0" --filter-name "FS200" \
	-filter "SOR > 3.0" --filter-name "SOR3" \
	-filter "MQ < 50.0" --filter-name "MQ50" \
	-filter "MQRankSum < -5.0" --filter-name "MQRankSum-50" \
	-filter "ReadPosRankSum < -5.0" --filter-name "ReadPosRankSum-5" \
	-O allsamples.step2.filtered.indels.vcf.gz

Juntando em um único VCF

In [ ]:
%%bash

	gatk MergeVcfs\
	-I allsamples.step2.filtered.snps.vcf.gz \
	-I allsamples.step2.filtered.indels.vcf.gz \
	-O allsamples.step2.filtered.variants.vcf.gz

Remova as variantes com flag de filtro

In [ ]:
%%bash

	gatk SelectVariants --exclude-filtered \
	-R genome.fasta \
	-V allsamples.step2.filtered.variants.vcf.gz \
	-O allsamples.step2.filtered.clean.variants.vcf.gz
	zgrep -v "^#" allsamples.step2.filtered.clean.variants.vcf.gz | wc -l

### 11. Correr FreeBayes 2 com os BAMs recalibrados
Gera uma lista de arquivos BAM para uso com FreeBayes (fofn = file of file names) e ativa o ambiente Conda contendo o FreeBayes. Roda o FreeBayes em paralelo, dividindo o genoma em 50 chunks para acelerar.

In [ ]:
%%bash

	ls *_recalibrated.step2.bam > recalibrated.step2.bams.fofn
	conda activate freebayes
	freebayes-parallel <(fasta_generate_regions.py genome.fasta.fai --chunks 50) 16    \
	-f genome.fasta \
	-g 200 \
	--bam-list recalibrated.step2.bams.fofn  > freebayes/freebayes.step2.raw.variants.vcf

Utilizando rtg-tools para obter estatísticas do VCF. Comprime com `bgzip` e indexa com `tabix` para uso posterior.

In [ ]:
%%bash

    ~/rtg-tools-3.12.1/rtg vcfstats freebayes.step2.raw.variants.vcf > freebayes.step2.raw.variants.rtg.stats
    bgzip freebayes.step2.raw.variants.vcf
    tabix freebayes.step2.raw.variants.vcf.gz

Extrai os valores de QUAL e DP das variantes com QUAL > 20 (https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9163752/)

In [ ]:
%%bash

	bcftools query --include 'QUAL>20' --format '%QUAL\n' freebayes.step2.raw.variants.vcf.gz > quals.step2.tmp
	bcftools query --include 'QUAL>20' --format '%INFO/DP\n' freebayes.step2.raw.variants.vcf.gz | awk -F "," '{print $1}' > dps.step2.tmp

Utilizando R para calcular

In [ ]:
%%R

	R
	 dps <- as.numeric(readLines("dps.step2.tmp")) 
	 quals <- as.numeric(readLines("quals.step2.tmp"))
	 qudp <- unname(round(quantile(dps,0.99)))
	 ququ <- unname(round(quantile(quals,0.99)))
	 write(qudp,file("dpt.step2.tmp"))
	 write(ququ,file("qut.step2.tmp"))

Filtragem, decomposição e normalização das variantes complexas. Foi necessário fazer downgrade do vcflib para a versão 1.0.1 no ambiente Conda do FreeBayes para que isso funcionasse.

In [ ]:
%%bash

	conda activate vcflib-env
	less qut.step2.tmp
	less dpt.step2.tmp
	bcftools view --include 'QUAL>20 & INFO/DP>10 & QUAL<3277 & INFO/DP<193 & (QUAL/(INFO/DP))>2' freebayes.step2.raw.variants.vcf.gz | vcfallelicprimitives -kg | bcftools norm --fasta-ref ../genome.fasta --output-type z --output freebayes.filtered.norm.step2.vcf.gz

	zgrep -v "^#" freebayes.filtered.norm.step2.vcf.gz | wc -l

Encontrando SNPs em comúm com GATK-step2 e Freebayes datasets, utilizando `bedtools`

In [ ]:
%%bash

	conda activate bedtools-env
	bedtools intersect -sorted -g genome.fasta.fai -header -u -a allsamples.step2.filtered.clean.variants.vcf.gz -b freebayes/freebayes.filtered.norm.step2.vcf.gz -wa > gatk.step2.freebayes.intersect.vcf
	
	zgrep -v "^#" gatk.step2.freebayes.intersect.vcf | wc -l

Selecionando variantes bialélicas com MAF > 0.05

In [ ]:
%%bash

	gatk IndexFeatureFile -I gatk.step2.freebayes.intersect.vcf
	gatk SelectVariants -R genome.fasta -V gatk.step2.freebayes.intersect.vcf --restrict-alleles-to BIALLELIC --exclude-non-variants -select "AF > 0.05" -O gatk.step2.freebayes.intersect.biallelic.maf0.05.vcf.gz

	zgrep -v "^#" gatk.step2.freebayes.intersect.biallelic.maf0.05.vcf.gz | wc -l

Reduz o número de variantes, mantendo apenas 1 variante a cada 150 pares de bases (pruning)

In [ ]:
%%bash

	bcftools +prune -n 1 -w 150bp --AF-tag "0.05" gatk.step2.freebayes.intersect.biallelic.maf0.05.vcf.gz -Oz -o gatk.step2.freebayes.biallelic.maf0.05.prune150.vcf.gz
	gatk IndexFeatureFile -I gatk.step2.freebayes.biallelic.maf0.05.prune150.vcf.gz

	zgrep -v "^#" gatk.step2.freebayes.biallelic.maf0.05.prune150.vcf.gz | wc -l

### 12. Informed pipeline - Variant Quality Score Recalibration 2 (VQSR2)
Converte o VCF em sítios apenas, removendo informações das amostras

In [ ]:
%%bash

	gatk  MakeSitesOnlyVcf \
	-I allsamples.step2.raw.snps4bqsr.vcf.gz \
	-O allsamples.step2.sitesonly.raw.snps4vqsr.vcf.gz

O arquivo em específico que será utilizado para o VQSR é o  `gatk.step2.freebayes.biallelic.maf0.05.prune150.vcf.gz` e já está indexado.
Criando um modelo estatístico (mixture of Gaussians) com 6 componentes para classificar variantes com base em anotações como: MQ, QD, FS, SOR, DP, ReadPosRankSum, etc. Define as `tranches` de confiança: variantes mais confiáveis são agrupadas em `tranches` superiores (ex: 99.9% de sensibilidade).

In [ ]:
%%bash

	gatk --java-options "-Xmx64g -Xms64g" VariantRecalibrator \
	-V allsamples.step2.sitesonly.raw.snps4vqsr.vcf.gz \
	--trust-all-polymorphic \
	-tranche 100.0 -tranche 99.9 -tranche 99.0 -tranche 97.5 -tranche 95.0 -tranche 90.0 \
	-an MQ -an MQRankSum -an QD -an ReadPosRankSum -an FS -an SOR -an DP \
	-mode SNP \
	--max-gaussians 6 \
	-resource:step2intersect,known=false,training=true,truth=true,prior=10.0 gatk.step2.freebayes.biallelic.maf0.05.prune150.vcf.gz \
	-O allsamples.step2.snps.recal.vcf.gz \
	--tranches-file aallsamples.step2.snps.tranches \
	--rscript-file allsamples.step2.vqsr.plots.R

Aplica VQSR2

In [ ]:
%%bash

	gatk --java-options "-Xmx64g -Xms64g" ApplyVQSR \
	-V allsamples.step2.raw.variants.vcf.gz \
	--recal-file allsamples.step2.snps.recal.vcf.gz \
	--tranches-file aallsamples.step2.snps.tranches \
	--truth-sensitivity-filter-level 90.0 \
	--create-output-variant-index true \
	-mode SNP \
	-O allsamples.step2.recalibrated.snps.vcf.gz

Selecionando tranche90

In [ ]:
%%bash

	gatk SelectVariants \
	--exclude-filtered \
	-R genome.fasta \
	-V allsamples.step2.recalibrated.snps.vcf.gz \
	-O allsamples.step2.tranche90.PASS.variants.vcf.gz

	zgrep -v "^#" allsamples.step2.tranche90.PASS.variants.vcf.gz | wc -l

Selecionando SNPs bialélicos com MAF > 0.05

In [ ]:
%%bash

	gatk SelectVariants -R genome.fasta \
	-V allsamples.step2.tranche90.PASS.variants.vcf.gz \
	-select-type SNP \
	--restrict-alleles-to BIALLELIC \
	--exclude-non-variants \
	-select "AF > 0.05" \
	-O allsamples.step2.tranche90.PASS.biallelic.snps.maf0.05.vcf.gz

	zgrep -v "^#" allsamples.step2.tranche90.PASS.biallelic.snps.maf0.05.vcf.gz | wc -l

Reduz o número de variantes, mantendo apenas 1 variante a cada 150 pares de bases (pruning)

In [ ]:
%%bash

	bcftools +prune -n 1 \
	-w 150bp \
	--AF-tag "0.05" \
	allsamples.step2.tranche90.PASS.biallelic.snps.maf0.05.vcf.gz \
	-Oz -o allsamples.step2.tranche90.PASS.biallelic.maf0.05.prune150.vcf.gz  

	zgrep -v "^#" allsamples.step2.tranche90.PASS.biallelic.maf0.05.prune150.vcf.gz | wc -l